<a href="https://colab.research.google.com/github/elhamod/IS834_Fall_2026/blob/main/Session%2005%20-%20Pandas%20III%20-%20Aggregation%20and%20Reshaping/05_Analytics_with_pandas_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS834 — Session 5

Work through this notebook in Colab. The lecture cells are filled in and ready to run; the
**Warmups** and **Exercises** are yours to complete — write your code in the `# Your code here` cells and your
reasoning in the `*Provide your answer here*` cells.

Every dataset is read straight from the course repository, so there is nothing to upload or install.

*Adapted from Brock Tibert's IS834 Fall 2025 session 5. Edited for Fall 2026 by Mohannad Elhamod.*

# Learning Goals

Sessions 3 and 4 got one dataset loaded, described and cleaned. All of that was preparation for a
question you cannot answer a row at a time: *how do the groups compare?*

- Group summaries with `groupby`
- `.count()` versus `.size()`, and why they disagree
- Aggregating several columns, and named aggregates
- Cross-tabulations and pivot tables
- Reshaping: wide to long and back with `.melt()`
- Choosing between `groupby`, `crosstab` and `pivot_table`

A look ahead:

- **Quiz 2** at the start of next session (session 6), handwritten, same format as Quiz 1
- Plotting and exploratory data analysis (session 6)
- A full end-to-end practice lab (session 7) — load, clean, aggregate, visualize, communicate
- Combining datasets with joins and merges (session 8)

# Setup


In [1]:
# pandas is the only package this notebook needs, and Colab already has it installed.
import pandas as pd

# Every data file this notebook uses is read straight from the course repository.
# There is nothing to upload: DATA is the folder those files live in.
DATA = "https://raw.githubusercontent.com/elhamod/IS834_Fall_2026/main/data/"


In [2]:
# Load the diamonds dataset, used in the practice exercise later in the session.
diamonds = pd.read_csv(DATA + "diamonds.csv")

print(diamonds.shape)

(53940, 11)


**Tip.** `%whos` lists every variable currently sitting in memory — useful when you lose track of what a notebook has already defined.

In [3]:
# List the variables currently in memory.
%whos

Variable   Type         Data/Info
---------------------------------
DATA       str          https://raw.githubusercon<...>S834_Fall_2026/main/data/
diamonds   DataFrame           rownames  carat   <...>[53940 rows x 11 columns]
pd         module       <module 'pandas' from '/u<...>ages/pandas/__init__.py'>


# Warmup Exercise!


**Warmup 1.** Read the `Orders` worksheet from the `sales.xlsx` workbook.

In [12]:
# Your code here
orders =  pd.read_excel(DATA + 'sales.xlsx', sheet_name='Orders')
orders.head()

,Row ID,Order Priority,Discount,Unit Price,Shipping Cost,Customer ID,Customer Name,Ship Mode,Customer Segment,Product Category,...,Region,State or Province,City,Postal Code,Order Date,Ship Date,Profit,Quantity ordered new,Sales,Order ID
0,18606,Not Specified,0.01,2.88,0.50,2,Janice Fletcher,Regular Air,Corporate,Office Supplies,...,Central,Illinois,Addison,60101,2012-05-28,2012-05-30,1.3200,2,5.90,88525
1,20847,High,0.01,2.84,0.93,3,Bonnie Potter,Express Air,Corporate,Office Supplies,...,West,Washington,Anacortes,98221,2010-07-07,2010-07-08,4.5600,4,13.01,88522
2,23086,Not Specified,0.03,6.68,6.15,3,Bonnie Potter,Express Air,Corporate,Office Supplies,...,West,Washington,Anacortes,98221,2011-07-27,2011-07-28,-47.6400,7,49.92,88523
3,23087,Not Specified,0.01,5.68,3.60,3,Bonnie Potter,Regular Air,Corporate,Office Supplies,...,West,Washington,Anacortes,98221,2011-07-27,2011-07-28,-30.5100,7,41.64,88523
4,23088,Not Specified,0.00,205.99,2.50,3,Bonnie Potter,Express Air,Corporate,Technology,...,West,Washington,Anacortes,98221,2011-07-27,2011-07-27,998.2023,8,1446.67,88523


**Warmup 2.** Return the information about the dataset. Are there any missing values?

In [13]:
# Your code here
orders.isna().sum()

,0
Row ID,0
Order Priority,0
Discount,0
Unit Price,0
Shipping Cost,0
Customer ID,0
Customer Name,0
Ship Mode,0
Customer Segment,0
Product Category,0


**Answer**

*Provide your answer here*

**Warmup 3.** Summarize the numeric variables in the dataset.

In [14]:
# Your code here
orders.describe()

,Row ID,Discount,Unit Price,Shipping Cost,Customer ID,Product Base Margin,Postal Code,Order Date,Ship Date,Profit,Quantity ordered new,Sales,Order ID
count,9426.000000,9426.000000,9426.000000,9426.000000,9426.000000,9354.000000,9426.000000,9426,9426,9426.000000,9426.000000,9426.000000,9426.000000
mean,20241.015277,0.049628,88.303686,12.795142,1738.422236,0.512189,52446.327286,2012-03-05 18:33:59.465308672,2012-03-07 19:16:18.485041408,139.236410,13.798430,949.706272,82318.489073
min,2.000000,0.000000,0.990000,0.490000,2.000000,0.350000,1001.000000,2010-01-01 00:00:00,2010-01-02 00:00:00,-16476.838000,1.000000,1.320000,6.000000
25%,19330.250000,0.020000,6.480000,3.192500,898.000000,0.380000,29406.000000,2011-03-07 06:00:00,2011-03-09 00:00:00,-74.017375,5.000000,61.282500,86737.250000
50%,21686.500000,0.050000,20.990000,6.050000,1750.000000,0.520000,52302.000000,2012-04-08 00:00:00,2012-04-09 00:00:00,2.567600,10.000000,203.455000,88344.500000
75%,24042.750000,0.080000,85.990000,13.990000,2578.750000,0.590000,78516.000000,2013-03-26 00:00:00,2013-03-28 00:00:00,140.243850,17.000000,776.402500,89987.750000
max,26399.000000,0.250000,6783.020000,164.730000,3403.000000,0.850000,99362.000000,2013-12-31 00:00:00,2014-01-17 00:00:00,16332.414000,170.000000,100119.160000,91591.000000
std,6101.890965,0.031798,281.540982,17.181203,979.167197,0.135229,29374.597802,NaN,NaN,998.486483,15.107688,2598.019818,19149.448857


**Answer**

*Provide your answer here*

> Session 3 answered "which region is struggling?" with four filtered DataFrames and a loop, and
> session 4 spent its time making a file trustworthy enough to ask the question of. Today we do the
> same summary in a single line — and then everything that a single line makes newly worth asking.

# Group Summaries


## GroupBy

`groupby` works in three steps: **split** the rows into groups by the values of a column, **apply** an aggregation (a count, a sum, a mean) to each group, and **combine** the results into one small table.

![](https://www.w3resource.com/w3r_images/pandas-groupby-split-apply-combine.svg)

The same idea with more than one grouping column — every combination of values becomes its own group:

![](https://miro.medium.com/v2/resize:fit:1400/0*m-puRNwcnUl5zZ4O)

**Tidy the column names first.** The warmup names have capitals and spaces, so `orders.Ship Mode` cannot work — a name with a space can only be reached with brackets and quotes. Renaming once at the top means every column can be written the same short way.

In [20]:
orders.columns

Index(['Row ID', 'Order Priority', 'Discount', 'Unit Price', 'Shipping Cost',
       'Customer ID', 'Customer Name', 'Ship Mode', 'Customer Segment',
       'Product Category', 'Product Sub-Category', 'Product Container',
       'Product Name', 'Product Base Margin', 'Region', 'State or Province',
       'City', 'Postal Code', 'Order Date', 'Ship Date', 'Profit',
       'Quantity ordered new', 'Sales', 'Order ID'],
      dtype='object')

In [21]:
# Lowercase the column names and replace spaces with underscores.
# Reading the worksheet again here means this cell works whether or not you finished Warmup 1.
orders = pd.read_excel(DATA + "sales.xlsx", sheet_name="Orders")
orders.columns = orders.columns.str.replace(" ", "_").str.lower()
orders.columns.tolist()

['row_id',
 'order_priority',
 'discount',
 'unit_price',
 'shipping_cost',
 'customer_id',
 'customer_name',
 'ship_mode',
 'customer_segment',
 'product_category',
 'product_sub-category',
 'product_container',
 'product_name',
 'product_base_margin',
 'region',
 'state_or_province',
 'city',
 'postal_code',
 'order_date',
 'ship_date',
 'profit',
 'quantity_ordered_new',
 'sales',
 'order_id']

`value_counts()` is the simple version of a groupby: it groups the rows by one column and counts how many fall in each group.

In [23]:
# Count the orders per ship mode.
x = orders.ship_mode.value_counts()
x

,count
ship_mode,
Regular Air,7036
Delivery Truck,1283
Express Air,1107


pandas gives both rows and columns an **index** — a set of labels. In the result above, the ship modes have become the row labels:

In [24]:
# The row labels (the index) of x.
x.index

Index(['Regular Air', 'Delivery Truck', 'Express Air'], dtype='object', name='ship_mode')

`groupby` is the general tool. But on its own it is "greedy": ask it to `.count()` and it counts every column it can.

In [25]:
# Group by ship mode and count -- every column at once.
orders.groupby("ship_mode").count()

,row_id,order_priority,discount,unit_price,shipping_cost,customer_id,customer_name,customer_segment,product_category,product_sub-category,...,region,state_or_province,city,postal_code,order_date,ship_date,profit,quantity_ordered_new,sales,order_id
ship_mode,,,,,,,,,,,,,,,,,,,,,
Delivery Truck,1283,1283,1283,1283,1283,1283,1283,1283,1283,1283,...,1283,1283,1283,1283,1283,1283,1283,1283,1283,1283
Express Air,1107,1107,1107,1107,1107,1107,1107,1107,1107,1107,...,1107,1107,1107,1107,1107,1107,1107,1107,1107,1107
Regular Air,7036,7036,7036,7036,7036,7036,7036,7036,7036,7036,...,7036,7036,7036,7036,7036,7036,7036,7036,7036,7036


> That is 23 columns of counts when we wanted one. It is not wrong, just unfocused — and on a wide table it is slow. **Always tell `groupby` which column you want aggregated.**


In [26]:
# Replicating value_counts: group by ship_mode, keep the ship_mode column, count rows, sort descending.
orders.groupby("ship_mode")['ship_mode'].count().sort_values(ascending=False)


,ship_mode
ship_mode,
Regular Air,7036
Delivery Truck,1283
Express Air,1107


### The aggregation you pick is a decision, not a formality

- `.count()` counts values that are **present**; `.size()` counts **rows**, missing or not.
- `.mean()`, `.median()`, `.min()`, `.max()`, `.sum()`, `.std()` all work the same way.
- pandas is built on top of numpy, so in examples online you will often see `np.mean` and friends passed in where a plain string like `"mean"` does the same job. Prefer the string — it is shorter and it keeps numpy out of your imports.

In [28]:
# Rows per ship mode (.size) next to margins recorded per ship mode (.count).
grouped = orders.groupby("ship_mode")

pd.DataFrame({'rows (size)': grouped.size(),
              'margins present (count)': grouped['product_base_margin'].count()})

,rows (size),margins present (count)
ship_mode,,
Delivery Truck,1283,1266
Express Air,1107,1096
Regular Air,7036,6992


In [39]:
orders.groupby('ship_mode')['product_base_margin'].agg(['size', 'count'])

,size,count
ship_mode,,
Delivery Truck,1283,1266
Express Air,1107,1096
Regular Air,7036,6992


**Read the two columns against each other.** They disagree by exactly the 72 missing product margins we found in the warmup. `.size()` answers "how many orders shipped this way?"; `.count()` answers "how many of those have a margin recorded?". Reaching for the wrong one gives you a denominator that is quietly too small, which is how a "percentage of orders" turns into a "percentage of orders where the field happened to be filled in".


By default the group labels become the **index** of the result. `as_index=False` keeps them as an ordinary column instead, and returns a DataFrame.

In [41]:
# Count orders per ship mode, keeping ship_mode as an ordinary column.
xdf = orders.groupby("ship_mode", as_index=False)['row_id'].count()
print(type(xdf))
xdf

<class 'pandas.core.frame.DataFrame'>


,ship_mode,row_id
0,Delivery Truck,1283
1,Express Air,1107
2,Regular Air,7036


In [50]:
xdf.loc[[1,2],'row_id']

,row_id
1,1107
2,7036


> This is the form to default to. With the group labels as a normal column, the result behaves like any other table — you can filter it, merge it, plot it or write it to Excel without first remembering `.reset_index()`.


**Exercise 1.** Group by shipping mode **and** customer segment, and count the orders in each combination. *Hint:* a single grouping column is passed as a string; several are passed as a list of strings.

In [ ]:
# Your code here


**Answer**

*Provide your answer here*

In [51]:
# Two grouping columns, two aggregated columns, one aggregation.
orders.groupby(["ship_mode", "customer_segment"], as_index=False)[['sales', 'discount']].mean()


,ship_mode,customer_segment,sales,discount
0,Delivery Truck,Consumer,2968.350570,0.049468
1,Delivery Truck,Corporate,3139.909548,0.049677
2,Delivery Truck,Home Office,2601.483579,0.051572
3,Delivery Truck,Small Business,2687.258945,0.048164
4,Express Air,Consumer,593.909775,0.046667
5,Express Air,Corporate,611.260905,0.049849
6,Express Air,Home Office,692.474719,0.049925
7,Express Air,Small Business,644.253545,0.053682
8,Regular Air,Consumer,654.855252,0.050447
9,Regular Air,Corporate,623.428061,0.050036


In [52]:
# Two grouping columns, two aggregated columns, one aggregation.
# x = orders.groupby(["ship_mode", "customer_segment"], as_index=False)[['sales', 'discount']].mean()
# x.loc[0]

x = orders.groupby(["ship_mode", "customer_segment"], as_index=True)[['sales', 'discount']].mean()
x.loc['Delivery Truck'].loc['Corporate']



,Corporate
sales,3139.909548
discount,0.049677


In [53]:
# .agg lets you ask for several aggregations at once: average and total sales per segment.
orders.groupby(["customer_segment"], as_index=False)['sales'].agg(['mean', 'sum'])


,customer_segment,mean,sum
0,Consumer,968.962629,1835215.22
1,Corporate,968.708465,3269391.07
2,Home Office,936.507785,2168952.03
3,Small Business,911.663770,1678373.00


In [59]:
orders.groupby(["customer_segment", "ship_mode"], as_index=False)[['sales', 'profit']].agg(['mean', 'sum'])

customer_segment       ship_mode        sales                  profit  \
                                            mean         sum        mean   
0          Consumer  Delivery Truck  2968.350570   780676.20  169.763140   
1          Consumer     Express Air   593.909775   131847.97  120.861757   
2          Consumer     Regular Air   654.855252   922691.05   95.869844   
3         Corporate  Delivery Truck  3139.909548  1460057.94  223.727230   
4         Corporate     Express Air   611.260905   243281.84  149.536097   
5         Corporate     Regular Air   623.428061  1566051.29  136.142555   
6       Home Office  Delivery Truck  2601.483579   777843.59  143.031958   
7       Home Office     Express Air   692.474719   184890.75  133.488073   
8       Home Office     Regular Air   689.267251  1206217.69  117.406676   
9    Small Business  Delivery Truck  2687.258945   687938.29  299.275333   
10   Small Business     Express Air   644.253545   141735.78  130.865330   
11   Small Business     Regular Air   621.757458   848698.93  154.629842   

                   
              sum  
0    44647.705733  
1    26831.309961  
2   135080.609655  
3   104033.162104  
4    59515.366658  
5   341990.099021  
6    42766.555368  
7    35641.315533  
8   205461.682914  
9    76614.485182  
10   28790.372522  
11  211069.734777

In [60]:
# Named aggregates: new_column = ("source_column", "aggregation").
(orders
 .groupby("customer_segment", as_index=False)
 .agg(avg_sales=("sales", "mean"),
      max_discount=("discount", "max"),
      order_count=("row_id", "count"))
 )

,customer_segment,avg_sales,max_discount,order_count
0,Consumer,968.962629,0.11,1894
1,Corporate,968.708465,0.21,3375
2,Home Office,936.507785,0.25,2316
3,Small Business,911.663770,0.17,1841


**Why named aggregates win.** The output columns are called `avg_sales`, `max_discount` and `order_count` rather than a stack of `mean`/`max` headers you have to decode, and you can mix different aggregations on different columns in one call.

The wrapping parentheses are a formatting trick, not syntax: they let you put each step of the chain on its own line so the sequence reads top to bottom. You will see this style everywhere in production pandas code.


## Tips and Tricks with Groupby

1.  `as_index=False` flattens the group labels into ordinary columns, including when you group by several columns. What it does not flatten is the column headers: ask `.agg()` for several functions on several columns and you get two-level headers (`sales`/`mean`, `sales`/`sum`, …). Named aggregates avoid that.
1.  Getting aggregates for multiple columns is great, but you don't necessarily need to think in large tasks.  Break up your needs into smaller units of work, and we can combine them together (combining tables with joins and merges is session 8)
1.  `pd.crosstab` lets us interact multiple columns by allowing us to specify row/column definitions, along with an aggregation
1.  `.pivot_table` give us much more control over how we can aggregate our data.  While groupby is extremely powerful, there may be times when you want to have more control over your aggregation output.

In [62]:
# crosstab: one variable down the rows, another across the columns, one aggregation in the cells.
pd.crosstab(index=orders.ship_mode, columns=orders.region,
            values=orders.sales, aggfunc="count")


region,Central,East,South,West
ship_mode,,,,
Delivery Truck,386,329,242,326
Express Air,322,282,241,262
Regular Air,2191,1678,1471,1696


In [64]:
orders.groupby(['ship_mode','region'])['sales'].sum()

ship_mode       region 
Delivery Truck  Central     974892.40
                East        989237.74
                South       655785.15
                West       1086600.73
Express Air     Central     196850.07
                East        225439.92
                South       113230.82
                West        166235.53
Regular Air     Central    1368599.15
                East       1208127.02
                South       828330.25
                West       1138602.54
Name: sales, dtype: float64

> Same warning as before, in a new outfit: `aggfunc="count"` counts orders that have a sales figure, while leaving `values`/`aggfunc` off entirely counts every row in the cell. On this column they agree because `sales` is complete — on `product_base_margin` they would not.


In [65]:
# pivot_table does the same job with more control -- several value columns at once, plus totals.
pd.pivot_table(orders, index="ship_mode", columns="region",
               values=["sales", "profit"], aggfunc="mean", margins=True).round(2)


profit                                    sales           \
region         Central    East   South    West     All  Central     East   
ship_mode                                                                  
Delivery Truck  228.07  234.46  141.96  210.23  208.93  2525.63  3006.80   
Express Air     167.14  139.73   65.84  159.11  136.20   611.34   799.43   
Regular Air     172.51  155.56   36.70  118.29  127.00   624.65   719.98   
All             179.31  164.95   53.33  136.10  139.24   876.28  1058.46   

                                           
region            South     West      All  
ship_mode                                  
Delivery Truck  2709.86  3333.13  2888.94  
Express Air      469.84   634.49   633.93  
Regular Air      563.11   671.35   645.77  
All              817.48  1047.04   949.71

> `margins=True` adds the "All" row and column. Note that the `All` figure is the mean over **all rows**, not the mean of the cells above it; averaging averages of unequal group sizes gives a different (and usually wrong) number.

**groupby, crosstab or pivot_table?** They compute the same numbers; they differ in shape. `groupby` returns one row per group — the long format that feeds plots and merges. `pivot_table` and `crosstab` return a grid — the wide format a human reads in a slide. `margins=True` adds the row and column totals that a director will ask for anyway.

Rule of thumb: **compute long with `groupby`, present wide with `pivot_table`.**


## Practice: the diamonds data

We are going to use the diamonds dataset loaded in the Setup section.

**Exercise 2.** For each combination of cut and color, get the average price and the number of rows, then sort so the most expensive combinations come first.

In [67]:
# Your code here
diamonds.groupby(['cut', 'color'])['price'].agg(['mean','count']).round(2).sort_values(by='mean', ascending=False)

mean  count
cut       color                
Premium   J      6294.59    808
          I      5946.18   1428
Very Good I      5255.88   1204
Premium   H      5216.71   2360
Fair      H      5135.68    303
Very Good J      5103.51    678
Good      I      5078.53    522
Fair      J      4975.66    119
Ideal     J      4918.19    896
Fair      I      4685.45    175
Good      J      4574.17    307
Very Good H      4535.39   1824
Premium   G      4500.74   2924
Ideal     I      4451.97   2093
Premium   F      4324.89   2331
Fair      D      4291.06    163
Good      H      4276.25    702
Fair      G      4239.25    314
Good      G      4123.48    871
Ideal     H      3889.33   3115
Very Good G      3872.75   2299
Fair      F      3827.00    312
Very Good F      3778.82   2164
Ideal     G      3720.71   4884
Fair      E      3682.31    224
Premium   D      3631.29   1603
          E      3538.91   2337
Good      F      3495.75    909
Very Good D      3470.47   1513
Good      E      3423.64    933
          D      3405.38    662
Ideal     F      3374.94   3826
Very Good E      3214.65   2400
Ideal     D      2629.09   2834
          E      2597.55   3903

In [77]:
diamonds.groupby(['cut','color']).agg(avg_sales=('price','mean')).round(2).sort_values(by='avg_sales', ascending=False)

avg_sales
cut       color           
Premium   J        6294.59
          I        5946.18
Very Good I        5255.88
Premium   H        5216.71
Fair      H        5135.68
Very Good J        5103.51
Good      I        5078.53
Fair      J        4975.66
Ideal     J        4918.19
Fair      I        4685.45
Good      J        4574.17
Very Good H        4535.39
Premium   G        4500.74
Ideal     I        4451.97
Premium   F        4324.89
Fair      D        4291.06
Good      H        4276.25
Fair      G        4239.25
Good      G        4123.48
Ideal     H        3889.33
Very Good G        3872.75
Fair      F        3827.00
Very Good F        3778.82
Ideal     G        3720.71
Fair      E        3682.31
Premium   D        3631.29
          E        3538.91
Good      F        3495.75
Very Good D        3470.47
Good      E        3423.64
          D        3405.38
Ideal     F        3374.94
Very Good E        3214.65
Ideal     D        2629.09
          E        2597.55

In [39]:
pd.crosstab(index=diamonds.cut, columns=diamonds.color,
            values=diamonds.price, aggfunc='mean').round(2).sort_values(by='D')

color,D,E,F,G,H,I,J
cut,,,,,,,
Ideal,2629.09,2597.55,3374.94,3720.71,3889.33,4451.97,4918.19
Good,3405.38,3423.64,3495.75,4123.48,4276.25,5078.53,4574.17
Very Good,3470.47,3214.65,3778.82,3872.75,4535.39,5255.88,5103.51
Premium,3631.29,3538.91,4324.89,4500.74,5216.71,5946.18,6294.59
Fair,4291.06,3682.31,3827.00,4239.25,5135.68,4685.45,4975.66


**Answer**

*Provide your answer here*

# Mini Practice Problem: Data Request


Your firm has just begun to formally track and collect customer complaint data. The tracking system and process are new, so the data is not yet perfect.

You are on the data team and assigned a Jira ticket to help the Director of Customer Support. The director wants your help to understand the main sources of complaints and is looking for guidance on what changes should be prioritized first.

__Question:__ Based on your analysis of the complaint data, what is your recommendation for where the manager should spend their time?


In [78]:
# Starter code: a list of dictionaries, which pd.DataFrame turns straight into a table.
data = [
    {"complaint_id": "C001", "category": "Card",    "description": "Charged twice",                "date_submitted": "2025-07-01", "tenure_months": 12,  "loyalty_program": "Silver",   "customer_spend_usd": 5000},
    {"complaint_id": "C002", "category": "Cards",   "description": "Card not received",            "date_submitted": "2025-07-02", "tenure_months": 24,  "loyalty_program": "Gold",     "customer_spend_usd": 12000},
    {"complaint_id": "C003", "category": "CRAD",    "description": "Card declined at POS",         "date_submitted": "2025-07-02", "tenure_months": None,"loyalty_program": None,       "customer_spend_usd": 2000},
    {"complaint_id": "C004", "category": "ATM",     "description": "Cash not dispensed",           "date_submitted": "2025-07-03", "tenure_months": 6,   "loyalty_program": "Bronze",   "customer_spend_usd": 1500},
    {"complaint_id": "C005", "category": "atm",     "description": "Machine out of service",       "date_submitted": "2025-07-03", "tenure_months": 8,   "loyalty_program": "Bronze",   "customer_spend_usd": 2000},
    {"complaint_id": "C006", "category": "Loans",   "description": "Late fee dispute",             "date_submitted": "2025-07-04", "tenure_months": None,"loyalty_program": None,       "customer_spend_usd": 800},
    {"complaint_id": "C007", "category": "Loan",    "description": "Payment not applied",          "date_submitted": "2025-07-05", "tenure_months": 18,  "loyalty_program": "Silver",   "customer_spend_usd": 7000},
    {"complaint_id": "C008", "category": "checking","description": "Unexpected monthly fee",       "date_submitted": "2025-07-06", "tenure_months": 3,   "loyalty_program": "Bronze",   "customer_spend_usd": 1000},
    {"complaint_id": "C009", "category": "Checking","description": "Overdraft fee issue",          "date_submitted": "2025-07-06", "tenure_months": None,"loyalty_program": None,       "customer_spend_usd": 500},
    {"complaint_id": "C010", "category": "Mortg.",  "description": "Escrow analysis confusion",    "date_submitted": "2025-07-07", "tenure_months": 30,  "loyalty_program": "Platinum", "customer_spend_usd": 25000},
    {"complaint_id": "C011", "category": "Mortgage","description": "Rate adjustment unclear",      "date_submitted": "2025-07-07", "tenure_months": 36,  "loyalty_program": "Platinum", "customer_spend_usd": 30000},
    {"complaint_id": "C012", "category": "Card",    "description": "Lost card replacement delay",  "date_submitted": "2025-07-08", "tenure_months": 5,   "loyalty_program": "Bronze",   "customer_spend_usd": 1200},
    {"complaint_id": "C013", "category": "ATM",     "description": "Deposit not posted",           "date_submitted": "2025-07-09", "tenure_months": None,"loyalty_program": None,       "customer_spend_usd": 700},
    {"complaint_id": "C014", "category": "Cards",   "description": "Incorrect card limit",         "date_submitted": "2025-07-10", "tenure_months": 20,  "loyalty_program": "Gold",     "customer_spend_usd": 11000},
    {"complaint_id": "C015", "category": "Loan",    "description": "Prepayment penalty confusion", "date_submitted": "2025-07-10", "tenure_months": 15,  "loyalty_program": "Silver",   "customer_spend_usd": 6000},
]

complaints = pd.DataFrame(data)
complaints


,complaint_id,category,description,date_submitted,tenure_months,loyalty_program,customer_spend_usd
0,C001,Card,Charged twice,2025-07-01,12.0,Silver,5000
1,C002,Cards,Card not received,2025-07-02,24.0,Gold,12000
2,C003,CRAD,Card declined at POS,2025-07-02,NaN,None,2000
3,C004,ATM,Cash not dispensed,2025-07-03,6.0,Bronze,1500
4,C005,atm,Machine out of service,2025-07-03,8.0,Bronze,2000
5,C006,Loans,Late fee dispute,2025-07-04,NaN,None,800
6,C007,Loan,Payment not applied,2025-07-05,18.0,Silver,7000
7,C008,checking,Unexpected monthly fee,2025-07-06,3.0,Bronze,1000
8,C009,Checking,Overdraft fee issue,2025-07-06,NaN,None,500
9,C010,Mortg.,Escrow analysis confusion,2025-07-07,30.0,Platinum,25000


The cleaning here is last session's work, so we do it in one cell and move on. The question this time
is not "what is in the file" but "what does the director do on Monday" -- and that is an aggregation
question, because it depends on how the complaints stack up by category.

In [79]:
# Step 1: last session's cleaning, compressed into one cell -- standardise the case, then map the
# typos onto one agreed spelling. Skip this and Card/Cards/CRAD count as three categories.
fixes = {'cards': 'card', 'crad': 'card', 'loans': 'loan', 'mortg.': 'mortgage'}

complaints['category_clean'] = complaints.category.str.lower().str.strip().replace(fixes)
complaints.category_clean.value_counts()

,count
category_clean,
card,5
atm,3
loan,3
checking,2
mortgage,2


In [89]:
# Your code here -- step 2: complaints and customer spend per category, in one table
z = complaints.groupby('category_clean').agg(customer_spend=('customer_spend_usd','sum'), complaints=('complaint_id','count'))
z

,customer_spend,complaints
category_clean,,
atm,4200,3
card,31200,5
checking,1500,2
loan,13800,3
mortgage,55000,2


In [90]:
# Your code here -- step 3: rank the categories a second way, by money rather than by volume
z.sort_values(by='customer_spend', ascending=False)

,customer_spend,complaints
category_clean,,
mortgage,55000,2
card,31200,5
loan,13800,3
atm,4200,3
checking,1500,2


In [101]:
# Your code here -- step 4: how much of the customer context (tenure, loyalty) is actually usable?
complaints.isna().sum()
# print(complaints.shape[0])

,0
complaint_id,0
category,0
description,0
date_submitted,0
tenure_months,4
loyalty_program,4
customer_spend_usd,0
category_clean,0


**Recommendation**

*Provide your answer here*

# Follow up Questions and Exercises

**Question 1.** `groupby` gives you the group labels as the index by default, and `as_index=False` turns them back into ordinary columns. Why does that choice matter in practice?


**Answer**

*Provide your answer here*

**Question 2.** `.count()` and `.size()` returned different numbers for the shipping-mode groups. Which one would you use to compute "the share of orders that shipped by Regular Air", and why?


**Answer**

*Provide your answer here*

**Question 3.** We produced the same numbers with `groupby`, `crosstab` and `pivot_table`. How would you choose between them?


**Answer**

*Provide your answer here*

**Exercise 1.** Reshaping. Build a wide table of total sales by region (rows) and product category (columns) for the Sales Director, then turn it back into a long table you could feed to a chart.


In [ ]:
# Your code here -- step 1: the wide table (regions down the side, product categories across the top)

In [ ]:
# Your code here -- step 2: turn the wide table back into a long one

**Answer**

*Provide your answer here*

**Exercise 2.** Grain. Check the orders table for duplicate rows, then check whether `order_id` uniquely identifies a row. What you find decides what a `groupby` on this table is actually counting.

In [ ]:
# Your code here


**Answer**

*Provide your answer here*

**Exercise 3.** Put groupby to work on the survey data: does income rise with education? Rebuild last session's ordered categorical column first, and say how much you trust the answer.

In [ ]:
# The survey data, freshly loaded.
df2 = pd.read_csv(DATA + "sample-survey-data.csv")

# Your code here -- step 1: mark the impossible income missing, and rebuild Education as an ordered categorical

In [ ]:
# Your code here -- step 2: respondents and average income per education level, in education order

**Answer**

*Provide your answer here*